# EDA: EX-UTR dataset

Exploratory data analysis of the UTR ↔ protein expression dataset.

**Goal:** Understand structure of the data before building models. Answer three questions:
1. What's the distribution of the target?
2. How are UTR sequences distributed (length, GC content)?
3. **What's the group structure of the data?** — critical for choosing validation strategy.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/expression_utr_summary.csv')
print(f'Shape: {df.shape}')
print(f'Unique genes: {df.gene_symbol.nunique()}')
print(f'Tissues: {df.tissue.nunique()}')
df.head()

## 1. Target distribution

In [ ]:
print(df['expression_level'].describe())

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df['expression_level'], bins=60, color='#1F3A5F', alpha=0.85)
ax.set_xlabel('log(expression + 1)')
ax.set_ylabel('count')
ax.set_title('Protein expression (log-scale)')
plt.show()

## 2. UTR lengths

In [ ]:
utr5_len = df['UTR5_Sequence'].str.len()
utr3_len = df['UTR3_Sequence'].str.len()
print(f'UTR5: median={utr5_len.median():.0f}, max={utr5_len.max()}')
print(f'UTR3: median={utr3_len.median():.0f}, max={utr3_len.max()}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(utr5_len, bins=50, color='#1F3A5F', alpha=0.85)
axes[0].set_title('UTR5 length')
axes[0].set_xlim(0, 800)
axes[1].hist(utr3_len, bins=50, color='#B25C48', alpha=0.85)
axes[1].set_title('UTR3 length')
axes[1].set_xlim(0, 3000)
plt.tight_layout()
plt.show()

## 3. Group structure — the critical question

Is UTR-sequence the same for all rows of one gene?

In [ ]:
n_utr5_per_gene = df.groupby('gene_symbol')['UTR5_Sequence'].nunique()
n_utr3_per_gene = df.groupby('gene_symbol')['UTR3_Sequence'].nunique()
print(f'Genes with a single unique UTR5: {(n_utr5_per_gene == 1).sum()} / {len(n_utr5_per_gene)}')
print(f'Genes with a single unique UTR3: {(n_utr3_per_gene == 1).sum()} / {len(n_utr3_per_gene)}')

**Result:** In 100% of cases, one gene has exactly one UTR5 and one UTR3. Expression varies across tissues; sequence does not. This is biologically correct — UTRs are DNA regions, invariant across tissues of the same individual.

**Implication for validation:** Random split by rows will place the same UTR-pair in both train and validation (for different tissues). The model won't need to generalize to new UTRs — it will "remember" them. See `docs/METHODOLOGY.md`.

## 4. Variance decomposition

How is the variance of `expression_level` distributed between differences across genes vs. differences across tissues within the same gene?

In [ ]:
gene_means = df.groupby('gene_symbol')['expression_level'].transform('mean')
total_var = df['expression_level'].var()
between_var = gene_means.var()
within_var = (df['expression_level'] - gene_means).var()
print(f'Total variance     : {total_var:.4f}')
print(f'Between-gene       : {between_var:.4f}  ({100*between_var/total_var:.1f}%)')
print(f'Within-gene        : {within_var:.4f}  ({100*within_var/total_var:.1f}%)')

**Interpretation:**
- 70% of variance is **between genes** — this is what UTR sequence *could* in principle predict.
- 30% of variance is **within gene** — tissue-driven, and by construction cannot depend on UTR sequence (which is identical across tissues of one gene).

So the ceiling for a pure UTR-based predictor (in R² terms, honestly evaluated with gene-level split) is bounded above by 70%, and in practice by how well UTR features actually correlate with per-gene mean expression.